One-vs-one (OvO) maps each class pair ``(a, b)`` to that pair's samples (the other classes are dropped). Pass the value source — ``df_parts`` (for ``CPP.run``) and/or ``dict_num_parts`` (for ``CPP.run_num``) — and OvO returns, per pair, the row-matched **copy** plus the binary labels, ready to drop into ``CPP`` (no mask to apply yourself):

In [1]:
import numpy as np
import pandas as pd
import aaanalysis as aa

labels = [0, 0, 1, 1, 2, 2]
df_parts = pd.DataFrame({"jmd_n": list("ACDEFG"), "tmd": list("HIKLMN"), "jmd_c": list("PQRSTV")})
# each pair -> (df_parts_pair, dict_num_parts_pair, labels_pair)
{pair: (len(dfp), y.tolist()) for pair, (dfp, _, y) in
 aa.SequenceFeature.get_labels_ovo(labels, df_parts=df_parts).items()}

{(0, 1): (4, [1, 1, 0, 0]),
 (0, 2): (4, [1, 1, 0, 0]),
 (1, 2): (4, [1, 1, 0, 0])}

Full OvO workflow: each pair already carries its row-matched ``df_parts``, so build a ``CPP`` on it directly, run, and collect:

In [2]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=9)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
multiclass = np.array([i % 3 for i in range(len(df_parts))])

frames = []
for (a, b), (df_pair, _, binary) in aa.SequenceFeature.get_labels_ovo(multiclass, df_parts=df_parts).items():
    df_feat = aa.CPP(df_parts=df_pair).run(labels=binary, n_filter=3)
    df_feat["pair"] = f"({a},{b})"
    frames.append(df_feat)
pd.concat(frames, ignore_index=True)[["pair", "feature", "abs_auc"]].head()

1. CPP creates 580140 features for 12 samples
1.1 Assigning scale values to parts
   |                         | 0.0%

   |........                 | 33.3%

   |................         | 66.7%

   |.........................| 100.0%


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

...


1.2 Streaming pre-filter stats (mask in stream)
   |                         | 0.0%

   |.........................| 100.0%

   |.........................| 100.0%
2. CPP pre-filters 29007 features (5.0%) with highest 'abs_mean_dif' and 'max_std_test' <= 0.2 (kept=516049 of 580140)


3. CPP filtering algorithm
4. CPP returns df of 3 unique features with general information and statistics


,pair,feature,abs_auc
0,"(0,1)","JMD_N_TMD_N-Pattern(C,4,8)-PALJ810106",0.5
1,"(0,1)","TMD-Pattern(N,5,9)-PALJ810106",0.5
2,"(0,1)","TMD-Segment(4,12)-PALJ810114",0.5
3,"(0,2)","JMD_N_TMD_N-Pattern(N,5,9)-AURR980115",0.5
4,"(0,2)","TMD-Pattern(N,4,7,11)-QIAN880105",0.5


**What can go wrong?** As for OvR, labels must be integers with more than one class:

In [3]:
try:
    aa.SequenceFeature.get_labels_ovo([1, 1, 1])
except ValueError as e:
    print("ValueError:", e)

ValueError: 'labels' should contain more than one different value ({np.int64(1)}).


**Further parameters.** Besides ``df_parts``, a numerical value source ``dict_num_parts`` (the per-part tensors consumed by :meth:`CPP.run_num`) is subset per pair too, and ``label_test`` / ``label_ref`` set which class of each pair becomes the positive (1) vs reference (0) label:

In [4]:
labels_multi = [0, 0, 1, 1, 2, 2]
df_parts_ex = pd.DataFrame({"tmd": list("HIKLMN")})
# per-part numerical tensor row-aligned to the labels: shape (n_samples, L_part, D)
dict_num_parts = {"tmd": np.random.default_rng(0).random((6, 4, 2))}
pairs = aa.SequenceFeature.get_labels_ovo(labels_multi, df_parts=df_parts_ex,
                                          dict_num_parts=dict_num_parts,
                                          label_test=1, label_ref=0)
# each pair -> subset (df_parts_pair, dict_num_parts_pair, labels_pair)
{pair: (dnp["tmd"].shape, y.tolist()) for pair, (_, dnp, y) in pairs.items()}

{(0, 1): ((4, 4, 2), [1, 1, 0, 0]),
 (0, 2): ((4, 4, 2), [1, 1, 0, 0]),
 (1, 2): ((4, 4, 2), [1, 1, 0, 0])}